# 02 — Column Normalization / Normalización de Columnas

**Project / Proyecto:** `excel-sales-consolidator`  
**Author / Autora:** Cristina  

---

**EN:** This notebook normalizes the column structure of the 10 source DataFrames loaded in `01_data_ingestion.ipynb`. It defines a standard column map, combines variant column names using `combine_first()`, adds a traceability column, and verifies that all DataFrames share the same structure before consolidation.

**ES:** Este notebook normaliza la estructura de columnas de los 10 DataFrames fuente cargados en `01_data_ingestion.ipynb`. Define un mapa de columnas estándar, combina variantes de nombres usando `combine_first()`, añade una columna de trazabilidad y verifica que todos los DataFrames comparten la misma estructura antes de la consolidación.

---

**Input:** `data/raw/source_01.xlsx` → `data/raw/source_10.xlsx`  
**Output:** `lista_dfs_normalized` — 10 DataFrames with identical column structure

---

### Pipeline contract / Contrato del pipeline

```python
# normalize_df(df, column_map, source_name) -> pd.DataFrame
# Each returned df has: 12 standard columns + source_file (traceability)
# normalize_dfs(list[df], column_map) -> list[df]  ← to extract as normalization.py
```

## 0 — Setup / Configuración

**EN:** Import required libraries, resolve the project root, and reload all 10 source files.  
**ES:** Importar librerías, resolver la raíz del proyecto y recargar los 10 archivos fuente.

In [ ]:
import pandas as pd
import sys
from pathlib import Path
from datetime import datetime

print("Libraries loaded successfully")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

ROOT = Path().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

lista_excel_files = [f"source_{i:02d}.xlsx" for i in range(1, 11)]

lista_dfs =[]

for f in lista_excel_files:
    INPUT_FILE = ROOT / "data" / "raw" / f
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_FILE}") 
        
    df = pd.read_excel(INPUT_FILE, dtype=str)  # Read all columns as string to avoid early type coercion
    df.columns = df.columns.str.strip()
    lista_dfs.append(df)

print(f"{len(lista_dfs)} files loaded successfully")          


## 1 — Column Map / Mapa de Columnas

**EN:** Define the mapping from all observed column name variants to 12 standard column names. Built manually from the full column inventory produced in `01_data_ingestion.ipynb`.  
**ES:** Define el mapeo de todas las variantes de nombres de columna observadas a 12 nombres de columna estándar. Construido manualmente a partir del inventario completo de columnas producido en `01_data_ingestion.ipynb`.

In [ ]:
COLUMN_MAP={
    "id_venta":       ["Venta ID", "ID_VENTA", "ID Venta", "Id venta"],
    "fecha_venta":    ["fecha venta", "fecha_venta_alt", "Fecha_Venta", "fecha operación", "fecha", "fecha_venta", "FECHA_VENTA", "Fecha Venta"],
    "producto":       ["Producto", "tipo_producto", "articulo", "producto"],
    "tipo_madera":    ["TIPO_MADERA", "tipo_madera", "madera", "tipo madera"],
    "certificacion":  ["certif", "certificacion", "certificación"],
    "cliente":        ["CLIENTE", "cliente", "Cliente", "cliente_2", "cliente_alt", "NOMBRE CLIENTE", "razon_social"],
    "cantidad_m3":    ["Cantidad m3", "cantidad_m3", "m3", "volumen_m3"],
    "precio_m3":      ["€/m3", "precio m3", "precio_m3", "Precio_m3"],
    "importe":        ["importe", "importe_total", "total"],
    "estado":         ["Estado", "estado", "situacion", "status"],
    "comercial":      ["comercial", "responsable_comercial", "sales_rep", "vendedor"],
    "pais":           ["country", "mercado", "pais", "País"]
}

## 2 — Normalization Function / Función de Normalización

**EN:** Define `normalize_df()` — the core function of this notebook. For each standard column it:
1. Finds which variants exist in the current DataFrame
2. If multiple variants exist → combines them with `combine_first()` to minimize data loss
3. If no variant exists → fills with `pd.NA`
4. Adds a `source_file` column for traceability

Column order is guaranteed by the order of keys in `COLUMN_MAP`.

**ES:** Define `normalize_df()` — la función central de este notebook. Para cada columna estándar:
1. Busca qué variantes existen en el DataFrame actual
2. Si hay varias variantes → las combina con `combine_first()` para minimizar pérdida de datos
3. Si no hay ninguna variante → rellena con `pd.NA`
4. Añade una columna `source_file` para trazabilidad

El orden de columnas está garantizado por el orden de claves en `COLUMN_MAP`.

In [ ]:

def normalize_df(df: pd.DataFrame, column_map: dict, source_name: str) -> pd.DataFrame:
    """
    EN: Maps raw column variants to standard names and keeps only relevant columns.
    ES: Mapea variantes de columnas al nombre estándar y conserva solo las relevantes.

    Args:
        df          : raw DataFrame from a single source file
        column_map  : dict mapping standard column names to lists of known variants
        source_name : filename string added as traceability column

    Returns:
        pd.DataFrame with exactly the standard columns + source_file, in consistent order
    """
    out = pd.DataFrame()

    for standard_col, variants in column_map.items():
       
        # Find which variants exist in this df
        
        found = [v for v in variants if v in df.columns]

        if not found:
            # No variant found — fill with NA to keep consistent structure
            out[standard_col] = pd.NA
        elif len(found) == 1:
            # Single variant — direct assignment
            out[standard_col] = df[found[0]]
        else:
            # Multiple variants — combine prioritizing first non-null value per row
            combined = df[found[0]]
            for col in found[1:]:
                combined = combined.combine_first(df[col])
            out[standard_col] = combined

    # Add traceability column to track which source file each row came from
    out["source_file"] = source_name

    return out.reset_index(drop=True)

## 3 — Apply Normalization / Aplicar Normalización

**EN:** Apply `normalize_df()` to all 10 DataFrames and collect results in `lista_dfs_normalized`.  
**ES:** Aplicar `normalize_df()` a los 10 DataFrames y recoger los resultados en `lista_dfs_normalized`.

In [ ]:
lista_dfs_normalized = []

for i, df in enumerate(lista_dfs, start=1):
    source_name = f"source_{i:02d}.xlsx"
    df_norm = normalize_df(df, COLUMN_MAP, source_name)
    lista_dfs_normalized.append(df_norm)
    print(f"✓ {source_name} — {df_norm.shape[0]} filas × {df_norm.shape[1]} columns")

print(f"\n{len(lista_dfs_normalized)} DataFrames normalized successfully")

## 4 — Verification / Verificación

**EN:** Confirm that all 10 DataFrames share exactly the same columns in the same order. This is the structural contract required for consolidation in the next step.  
**ES:** Confirmar que los 10 DataFrames comparten exactamente las mismas columnas en el mismo orden. Este es el contrato estructural necesario para la consolidación en el siguiente paso.

In [ ]:
# Verify all DataFrames share the same column structure
reference_cols = list(lista_dfs_normalized[0].columns)
all_equal = all(list(df.columns) == reference_cols for df in lista_dfs_normalized)

print(f"Column structure consistent across all files: {'✓ YES' if all_equal else '✗ NO — check mismatches below'}\n")
print(f"Standard columns ({len(reference_cols)}):")
for col in reference_cols:
    print(f"  · {col}")

In [ ]:
for i, df in enumerate(lista_dfs_normalized, start=1):
    print(f"source_{i:02d}: {list(df.columns)}")

In [ ]:
---

## Summary / Resumen

**EN:**
- `COLUMN_MAP` built from 53 unique column name variants across 10 source files
- `normalize_df()` applies mapping, `combine_first()` merging, and traceability in one pass
- All 10 DataFrames now share 13 columns (12 standard + `source_file`) in consistent order
- Ready for concatenation into a single consolidated DataFrame
- → Next step: `03_consolidation.ipynb`

**ES:**
- `COLUMN_MAP` construido a partir de 53 variantes únicas de nombres de columna en 10 archivos fuente
- `normalize_df()` aplica el mapeo, la combinación con `combine_first()` y la trazabilidad en un solo paso
- Los 10 DataFrames comparten ahora 13 columnas (12 estándar + `source_file`) en orden consistente
- Listos para la concatenación en un único DataFrame consolidado
- → Siguiente paso: `03_consolidation.ipynb`